# Event Tree Analysis (ETA) of a Cyber-Induced Failure in a Smart Power Grid Substation

## Brief Description of the Case Study
This project presents an Event Tree Analysis (ETA) for a smart power grid substation affected by a malicious or faulty control command. The case combines cybersecurity, physical protection systems, and human response in one integrated model.

The initiating event is:
**Malicious or faulty control command affects substation operation**

After this initiating event, the system may either recover safely or progress toward partial failure or catastrophic failure depending on the success or failure of multiple barriers.

## Barriers Considered in the Model
The event tree includes six defensive layers:

1. Intrusion Detection System (cybersecurity barrier)
2. Access Control / Authentication (cybersecurity barrier)
3. Protection Relay Logic (technical safety barrier)
4. Alarm / Supervisory Response (technical safety barrier)
5. Operator Intervention (human barrier)
6. Shutdown / Isolation System (technical safety barrier)

## What This Code Will Do
In this notebook, we will:

- define the ETA data structure
- create helper functions for probability formatting and outcome classification
- generate simple custom icons for ETA nodes
- build the Event Tree class
- define case-specific outcome logic for the smart substation
- create the case study with assigned barrier probabilities
- generate all possible event paths
- calculate path probabilities and outcome probabilities
- visualize the ETA diagram
- display summary tables for barriers, paths, final outcomes, and category comparisons

The final outputs will help explain how one cyber-induced initiating event can propagate through multiple protection layers and lead to different consequences.

## Block 1: Import Required Libraries

This block imports all libraries needed for the ETA model.

- `dataclass` is used to create clean data structures for barriers and path results.
- `typing` helps define expected input and output types.
- `os` and `tempfile` are used for file handling and temporary image storage.
- `defaultdict` is used for grouping outcome probabilities.
- `pandas` is used to create output tables.
- `matplotlib` is used to generate simple custom icons.
- `graphviz` is used to draw the event tree.
- `IPython.display` is used to display the generated ETA image inside the notebook.

In [8]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Callable
import os
import tempfile
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, FancyBboxPatch
from graphviz import Digraph
from IPython.display import Image, display

## Block 2: Define Core Data Classes

This block defines the main data structures used in the ETA model.

### ETABarrier
This class stores information about each barrier:
- barrier name
- success probability
- barrier type

It also checks whether the probability and type are valid.

### ETAPathResult
This class stores the result of one full event path:
- path ID
- sequence of success/failure decisions
- full path probability
- outcome name
- outcome category

These classes make the code more organized and readable.

In [18]:
# This decorator automatically creates init, repr, etc. for the class
@dataclass
class ETABarrier:
    # Name of the barrier (e.g., IDS, Authentication, Relay)
    name: str

    # Probability that the barrier successfully works
    success_probability: float

    # Type of barrier: SECURITY, SAFETY, or HUMAN (default is SAFETY)
    barrier_type: str = "SAFETY"

    # This function checks if the barrier inputs are valid
    def validate(self) -> None:

        # Ensure probability is between 0 and 1
        if not (0.0 <= self.success_probability <= 1.0):
            raise ValueError(
                f"Barrier '{self.name}' must have success probability between 0 and 1."
            )

        # Define allowed barrier categories
        valid_types = {"SECURITY", "SAFETY", "HUMAN"}

        # Check if the given barrier type is valid
        if self.barrier_type not in valid_types:
            raise ValueError(
                f"Barrier '{self.name}' has invalid type '{self.barrier_type}'. "
                f"Choose from {valid_types}."
            )


# This dataclass stores the result of one complete path in the event tree
@dataclass
class ETAPathResult:

    # Unique identifier for each path
    path_id: int

    # List of True/False values representing Success/Failure of each barrier
    decisions: List[bool]

    # Final probability of this entire path (product of all branches)
    path_probability: float

    # Description of the final outcome for this path
    outcome_name: str

    # Category of the outcome (SAFE / PARTIAL FAILURE / CATASTROPHIC FAILURE)
    outcome_category: str

## Block 3: Define Helper Functions

This block contains small utility functions used throughout the ETA model.

### Functions included
- `failure_probability()` converts success probability into failure probability
- `format_prob()` formats probabilities for readable output
- `branch_text()` converts True/False into Success/Failure
- `wrap_text()` wraps long text so labels fit better inside nodes
- `classify_outcome()` assigns each outcome to one of three categories:
  - SAFE
  - PARTIAL FAILURE
  - CATASTROPHIC FAILURE
- `outcome_fillcolor()` selects node color based on severity
- `outcome_icon_key()` selects the correct icon for final outcomes

These functions improve readability and make the later code cleaner.

In [19]:
# This function calculates failure probability from success probability
def failure_probability(success_probability: float) -> float:
    # Failure = 1 - Success
    return 1.0 - success_probability


# This function formats probability values for display
def format_prob(p: float, decimals: int = 4) -> str:
    # Rounds the value and formats it like "P = 0.1234"
    return f"P = {round(p, decimals):g}"


# This function converts True/False into readable text
def branch_text(decision: bool) -> str:
    # True means Success, False means Failure
    return "Success" if decision else "Failure"


# This function wraps long text so it fits nicely inside diagram boxes
def wrap_text(text: str, max_len: int = 26, max_lines: int = 5) -> str:

    # Split sentence into words
    words = text.split()

    # Store final wrapped lines
    lines = []

    # Temporary line being built
    current = ""

    # Loop through each word and build lines
    for word in words:
        # Try adding the next word to current line
        candidate = f"{current} {word}".strip()

        # If it fits within max length, keep adding
        if len(candidate) <= max_len:
            current = candidate
        else:
            # Otherwise, push current line and start a new one
            if current:
                lines.append(current)
            current = word

    # Add the last line if anything remains
    if current:
        lines.append(current)

    # If too many lines, trim and add "..."
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1][:max(0, max_len - 3)] + "..."

    # Join lines with newline characters for display
    return "\n".join(lines)


# This function classifies outcomes into categories
def classify_outcome(outcome_name: str) -> str:

    # Convert to lowercase for easier matching
    lower_name = outcome_name.lower()

    # If keywords indicate safe condition
    if any(k in lower_name for k in ["safe", "stable", "maintained", "isolated", "rejected"]):
        return "SAFE"

    # If keywords indicate partial or degraded condition
    if any(k in lower_name for k in ["partial", "delayed", "contained", "recovery", "local outage", "degradation"]):
        return "PARTIAL FAILURE"

    # Otherwise, treat it as catastrophic
    return "CATASTROPHIC FAILURE"


# This function assigns colors to outcomes for visualization
def outcome_fillcolor(category: str) -> str:

    # Green for safe outcomes
    if category == "SAFE":
        return "#D9EAD3"

    # Yellow for partial failures
    if category == "PARTIAL FAILURE":
        return "#FFF2CC"

    # Red for catastrophic failures
    return "#F4CCCC"


# This function selects which icon to use for each outcome type
def outcome_icon_key(category: str) -> str:

    # Safe icon
    if category == "SAFE":
        return "SAFE"

    # Partial failure icon
    if category == "PARTIAL FAILURE":
        return "PARTIAL"

    # Hazard icon for catastrophic cases
    return "HAZARD"

## Block 4: Create Custom Icons for the ETA Diagram

This block creates simple custom icons for:
- initiating event
- security barrier
- safety barrier
- human barrier
- safe outcome
- partial failure outcome
- hazard outcome

These icons are generated using matplotlib and then saved as image files.
Later, Graphviz uses these icons inside the event tree to improve visual clarity.

This step is mainly for better presentation and readability of the final ETA diagram.

In [20]:
# This function creates small icon images used in the ETA diagram
def create_eta_icons(output_dir: str) -> Dict[str, str]:

    # Create the output folder if it does not already exist
    os.makedirs(output_dir, exist_ok=True)

    # Store file paths for each icon that will be created
    paths = {
        "INIT": os.path.join(output_dir, "init_event.png"),
        "SECURITY": os.path.join(output_dir, "security_barrier.png"),
        "SAFETY": os.path.join(output_dir, "safety_barrier.png"),
        "HUMAN": os.path.join(output_dir, "human_barrier.png"),
        "SAFE": os.path.join(output_dir, "safe_outcome.png"),
        "PARTIAL": os.path.join(output_dir, "partial_outcome.png"),
        "HAZARD": os.path.join(output_dir, "hazard_outcome.png"),
    }

    # Create the initiating event icon figure
    fig, ax = plt.subplots(figsize=(2.3, 1.0))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes because this is only an icon
    ax.axis("off")

    # Draw a rounded box for the initiating event icon
    init_box = FancyBboxPatch(
        (0.08, 0.20), 0.84, 0.60,
        boxstyle="round,pad=0.03,rounding_size=0.05",
        facecolor="#F4CCCC",
        edgecolor="black",
        linewidth=1.2
    )

    # Add the rounded box to the figure
    ax.add_patch(init_box)

    # Add the label text inside the initiating event icon
    ax.text(0.5, 0.5, "INIT", ha="center", va="center", fontsize=11, weight="bold")

    # Save the initiating event icon as a PNG file
    plt.savefig(paths["INIT"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure to free memory
    plt.close(fig)

    # Create the security barrier icon figure
    fig, ax = plt.subplots(figsize=(1.4, 1.4))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a circle for the security icon
    obj = Circle((0.5, 0.5), 0.34, facecolor="#F4CCCC", edgecolor="black", linewidth=1.2)

    # Add the circle to the figure
    ax.add_patch(obj)

    # Add the short label inside the icon
    ax.text(0.5, 0.5, "SEC", ha="center", va="center", fontsize=8.5, weight="bold")

    # Save the security icon
    plt.savefig(paths["SECURITY"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Create the safety barrier icon figure
    fig, ax = plt.subplots(figsize=(1.4, 1.4))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a circle for the safety icon
    obj = Circle((0.5, 0.5), 0.34, facecolor="#D9EAD3", edgecolor="black", linewidth=1.2)

    # Add the circle to the figure
    ax.add_patch(obj)

    # Add the short label inside the icon
    ax.text(0.5, 0.5, "SAFE", ha="center", va="center", fontsize=7.5, weight="bold")

    # Save the safety icon
    plt.savefig(paths["SAFETY"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Create the human barrier icon figure
    fig, ax = plt.subplots(figsize=(1.4, 1.4))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a circle for the human action icon
    obj = Circle((0.5, 0.5), 0.34, facecolor="#CFE2F3", edgecolor="black", linewidth=1.2)

    # Add the circle to the figure
    ax.add_patch(obj)

    # Add the label inside the icon
    ax.text(0.5, 0.5, "H", ha="center", va="center", fontsize=11, weight="bold")

    # Save the human icon
    plt.savefig(paths["HUMAN"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Create the safe outcome icon figure
    fig, ax = plt.subplots(figsize=(2.0, 1.0))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a rectangle for the safe outcome icon
    obj = Rectangle((0.10, 0.18), 0.80, 0.64, facecolor="#D9EAD3", edgecolor="black", linewidth=1.2)

    # Add the rectangle to the figure
    ax.add_patch(obj)

    # Add the label inside the icon
    ax.text(0.5, 0.5, "SAFE", ha="center", va="center", fontsize=11, weight="bold")

    # Save the safe outcome icon
    plt.savefig(paths["SAFE"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Create the partial failure outcome icon figure
    fig, ax = plt.subplots(figsize=(2.1, 1.0))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a rectangle for the partial failure icon
    obj = Rectangle((0.08, 0.18), 0.84, 0.64, facecolor="#FFF2CC", edgecolor="black", linewidth=1.2)

    # Add the rectangle to the figure
    ax.add_patch(obj)

    # Add the label inside the icon
    ax.text(0.5, 0.5, "PARTIAL", ha="center", va="center", fontsize=10, weight="bold")

    # Save the partial outcome icon
    plt.savefig(paths["PARTIAL"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Create the hazard outcome icon figure
    fig, ax = plt.subplots(figsize=(2.1, 1.0))

    # Set x-axis range
    ax.set_xlim(0, 1)

    # Set y-axis range
    ax.set_ylim(0, 1)

    # Hide axes
    ax.axis("off")

    # Draw a rectangle for the hazard outcome icon
    obj = Rectangle((0.08, 0.18), 0.84, 0.64, facecolor="#EA9999", edgecolor="black", linewidth=1.2)

    # Add the rectangle to the figure
    ax.add_patch(obj)

    # Add the label inside the icon
    ax.text(0.5, 0.5, "HAZARD", ha="center", va="center", fontsize=10, weight="bold")

    # Save the hazard outcome icon
    plt.savefig(paths["HAZARD"], dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.02)

    # Close the figure
    plt.close(fig)

    # Return all saved icon file paths so they can be used later in the ETA diagram
    return paths

## Block 5: Build the Main EventTree Class

This is the central part of the project.

The `EventTree` class is responsible for:

- checking whether the tree input is valid
- generating all possible success/failure paths
- assigning final outcomes to each path
- calculating path probabilities
- generating output tables
- drawing the ETA diagram
- printing a dashboard summary

This class is the engine of the full ETA model.

In [21]:
# Main class that builds and manages the full Event Tree Analysis
class EventTree:

    # Constructor method to initialize the event tree object
    def __init__(
        self,
        initiating_event: str,                     # Starting event of the ETA
        barriers: List[ETABarrier],               # List of barriers in sequence
        title: str = "Event Tree Analysis",       # Default title for the analysis
        outcome_logic: Optional[Callable[[List[bool]], str]] = None  # Optional custom outcome logic
    ) -> None:

        # Store the initiating event text
        self.initiating_event = initiating_event

        # Store the list of barriers
        self.barriers = barriers

        # Store the title of the analysis
        self.title = title

        # Store the custom outcome logic function, if provided
        self.outcome_logic = outcome_logic

        # Validate the tree inputs as soon as the object is created
        self._validate_tree()

    # Internal method to check whether the event tree input is valid
    def _validate_tree(self) -> None:

        # Make sure the initiating event is not empty
        if not self.initiating_event.strip():
            raise ValueError("Initiating event must not be empty.")

        # Make sure at least one barrier exists
        if not self.barriers:
            raise ValueError("At least one barrier is required in the event tree.")

        # Validate each barrier one by one
        for barrier in self.barriers:
            barrier.validate()

    # Internal method to decide the final outcome for a given path
    def _determine_outcome(self, decisions: List[bool]) -> Tuple[str, str]:

        # If user has provided custom logic, use that first
        if self.outcome_logic is not None:
            outcome_name = self.outcome_logic(decisions)

            # Convert outcome description into a severity category
            outcome_category = classify_outcome(outcome_name)

            # Return both description and category
            return outcome_name, outcome_category

        # Default logic: if all barriers succeed, system stays safe
        if all(decisions):
            return "All barriers succeed - Safe state maintained", "SAFE"

        # If last barrier succeeds after earlier problems, treat as partial recovery
        if decisions[-1]:
            return "Late recovery after earlier failures", "PARTIAL FAILURE"

        # Otherwise, treat as catastrophic progression
        return "Barrier failure sequence leads to catastrophic state", "CATASTROPHIC FAILURE"

    # This method generates all possible success/failure paths in the tree
    def generate_paths(self) -> List[ETAPathResult]:

        # List to store final path results
        results: List[ETAPathResult] = []

        # Counter to assign unique path IDs
        path_counter = 1

        # Recursive helper function to explore all branches
        def recurse(index: int, current_decisions: List[bool], current_probability: float) -> None:

            # Allows updating the outer path counter variable
            nonlocal path_counter

            # If all barriers have been processed, finalize this path
            if index == len(self.barriers):
                outcome_name, outcome_category = self._determine_outcome(current_decisions)

                # Store the completed path result
                results.append(
                    ETAPathResult(
                        path_id=path_counter,
                        decisions=current_decisions.copy(),
                        path_probability=current_probability,
                        outcome_name=outcome_name,
                        outcome_category=outcome_category
                    )
                )

                # Move to the next path number
                path_counter += 1
                return

            # Get the current barrier being evaluated
            barrier = self.barriers[index]

            # First branch: assume this barrier succeeds
            current_decisions.append(True)
            recurse(index + 1, current_decisions, current_probability * barrier.success_probability)
            current_decisions.pop()

            # Second branch: assume this barrier fails
            current_decisions.append(False)
            recurse(index + 1, current_decisions, current_probability * failure_probability(barrier.success_probability))
            current_decisions.pop()

        # Start recursion from the first barrier with probability 1
        recurse(0, [], 1.0)

        # Return all generated paths
        return results

    # Create a table showing each barrier and its probabilities
    def barrier_table(self) -> pd.DataFrame:

        # Store table rows here
        rows = []

        # Loop through all barriers with numbering starting from 1
        for i, barrier in enumerate(self.barriers, start=1):
            rows.append({
                "ID": i,
                "Barrier": barrier.name,
                "Barrier Type": barrier.barrier_type,
                "Success Probability": format_prob(barrier.success_probability),
                "Failure Probability": format_prob(failure_probability(barrier.success_probability))
            })

        # Convert list of rows into a pandas DataFrame
        return pd.DataFrame(rows)

    # Create a table showing every complete path and its outcome
    def path_table(self) -> pd.DataFrame:

        # Store table rows here
        rows = []

        # Generate all possible paths and loop through them
        for result in self.generate_paths():

            # Build readable branch decision text for this path
            decision_text = " | ".join(
                f"{self.barriers[i].name}: {branch_text(result.decisions[i])}"
                for i in range(len(self.barriers))
            )

            # Add this path as one row in the table
            rows.append({
                "Path ID": result.path_id,
                "Branch Decisions": decision_text,
                "Path Probability": format_prob(result.path_probability),
                "Outcome": result.outcome_name,
                "Outcome Category": result.outcome_category
            })

        # Convert list of rows into a DataFrame
        return pd.DataFrame(rows)

    # Create a summary table by grouping identical outcomes together
    def outcome_summary_table(self) -> pd.DataFrame:

        # Dictionary to sum probabilities of same outcomes
        grouped = defaultdict(float)

        # Go through every generated path
        for result in self.generate_paths():
            grouped[(result.outcome_name, result.outcome_category)] += result.path_probability

        # Store final grouped rows here
        rows = []

        # Sort grouped outcomes by highest probability first
        for i, ((outcome_name, category), total_prob) in enumerate(
            sorted(grouped.items(), key=lambda x: x[1], reverse=True),
            start=1
        ):
            rows.append({
                "ID": i,
                "Outcome": outcome_name,
                "Outcome Category": category,
                "Total Probability": format_prob(total_prob)
            })

        # Return grouped outcome table
        return pd.DataFrame(rows)

    # Create a comparison table for SAFE, PARTIAL FAILURE, and CATASTROPHIC FAILURE totals
    def category_comparison_table(self) -> pd.DataFrame:

        # Dictionary to sum probabilities category-wise
        totals = defaultdict(float)

        # Add each path probability to its category total
        for result in self.generate_paths():
            totals[result.outcome_category] += result.path_probability

        # Fixed order of categories for presentation
        categories = ["SAFE", "PARTIAL FAILURE", "CATASTROPHIC FAILURE"]

        # Store rows for the final category table
        rows = []

        # Build one row per category
        for i, category in enumerate(categories, start=1):
            rows.append({
                "ID": i,
                "Category": category,
                "Total Probability": format_prob(totals.get(category, 0.0))
            })

        # Return the final category comparison table
        return pd.DataFrame(rows)

## Block 6: Add Visualization and Dashboard Methods

This block continues the `EventTree` class by adding:

### `visualize_event_tree()`
This method creates the event tree diagram using Graphviz.
It places:
- the initiating event
- all barriers
- success and failure branches
- final outcomes with colors and icons

### `run_dashboard()`
This method prints a summary and returns all final output tables.

This is the part that produces the final readable output of the project.

In [22]:
# This extends the earlier EventTree class by adding visualization and dashboard methods
class EventTree(EventTree):

    # This method draws the full ETA diagram and saves it as an image
    def visualize_event_tree(self, save_path: Optional[str] = None) -> str:

        # Use the provided file name, or fall back to a default name
        output_name = save_path if save_path else "eta_event_tree"

        # Create a temporary folder to store the generated icons
        temp_dir = tempfile.mkdtemp(prefix="eta_icons_")

        # Generate the icon images and get their file paths
        icon_paths = create_eta_icons(temp_dir)

        # Create a new Graphviz directed graph for the ETA
        dot = Digraph("ETA", format="png")

        # Set overall layout settings for a left-to-right tree
        dot.attr(rankdir="LR", splines="polyline", nodesep="0.75", ranksep="1.0")

        # Set default font for the graph
        dot.attr(fontname="Helvetica")

        # Set default font for nodes
        dot.attr("node", fontname="Helvetica")

        # Set default font for edges and branch labels
        dot.attr("edge", fontname="Helvetica", fontsize="9")

        # Define IDs for the initiating event text box and its icon
        init_event_id = "init_event"
        init_symbol_id = "init_symbol"

        # Add the initiating event text box
        dot.node(
            init_event_id,
            label=wrap_text(self.initiating_event, max_len=28, max_lines=4),
            shape="box",
            style="rounded,filled",
            fillcolor="white",
            color="black",
            width="2.8",
            height="0.95",
            penwidth="1.2"
        )

        # Add the initiating event icon next to the text box
        dot.node(
            init_symbol_id,
            label="",
            shape="none",
            image=icon_paths["INIT"],
            imagescale="true",
            width="1.1",
            height="0.6",
            fixedsize="true"
        )

        # Connect the initiating event box to its icon
        dot.edge(init_event_id, init_symbol_id, arrowhead="none", penwidth="1.1")

        # Start the tree from the initiating event icon
        current_nodes = [init_symbol_id]

        # Loop through each barrier level in the event tree
        for level, barrier in enumerate(self.barriers, start=1):

            # Store the nodes that will become parents for the next level
            next_nodes = []

            # For each current branch endpoint, place this barrier
            for parent_idx, parent_node in enumerate(current_nodes):

                # Create unique node IDs for the barrier text box and icon
                barrier_event_id = f"b{level}_{parent_idx}_event"
                barrier_symbol_id = f"b{level}_{parent_idx}_symbol"

                # Add the barrier text box
                dot.node(
                    barrier_event_id,
                    label=wrap_text(barrier.name, max_len=26, max_lines=4),
                    shape="box",
                    style="rounded,filled",
                    fillcolor="white",
                    color="black",
                    width="2.6",
                    height="0.85",
                    penwidth="1.1"
                )

                # Add the barrier icon based on its type
                dot.node(
                    barrier_symbol_id,
                    label="",
                    shape="none",
                    image=icon_paths[barrier.barrier_type],
                    imagescale="true",
                    width="0.78",
                    height="0.56",
                    fixedsize="true"
                )

                # Connect previous branch endpoint to barrier text box
                dot.edge(parent_node, barrier_event_id, arrowhead="none", penwidth="1.0")

                # Connect barrier text box to barrier icon
                dot.edge(barrier_event_id, barrier_symbol_id, arrowhead="none", penwidth="1.0")

                # Create IDs for the success and failure branch endpoints
                success_stub = f"b{level}_{parent_idx}_S"
                failure_stub = f"b{level}_{parent_idx}_F"

                # Add a simple label node for the success branch
                dot.node(success_stub, label="Success", shape="plaintext", fontcolor="darkgreen", fontsize="10")

                # Add a simple label node for the failure branch
                dot.node(failure_stub, label="Failure", shape="plaintext", fontcolor="red", fontsize="10")

                # Format success probability for display
                s_prob = format_prob(barrier.success_probability)

                # Calculate and format failure probability for display
                f_prob = format_prob(failure_probability(barrier.success_probability))

                # Draw the success edge with label and color
                dot.edge(
                    barrier_symbol_id,
                    success_stub,
                    label=f"S\n{s_prob}",
                    color="darkgreen",
                    penwidth="1.1"
                )

                # Draw the failure edge with label and color
                dot.edge(
                    barrier_symbol_id,
                    failure_stub,
                    label=f"F\n{f_prob}",
                    color="red",
                    penwidth="1.1"
                )

                # Add both branch endpoints to the next level
                next_nodes.extend([success_stub, failure_stub])

            # Move to the next level of the tree
            current_nodes = next_nodes

        # Generate all final path results
        all_paths = self.generate_paths()

        # Loop through each final path and place its outcome node
        for idx, result in enumerate(all_paths):

            # Get the branch endpoint that leads to this outcome
            parent_stub = current_nodes[idx]

            # Create a unique outcome node ID
            outcome_id = f"outcome_{idx}"

            # Build the final label shown inside the outcome box
            label_text = (
                f"Path {result.path_id}\n"
                f"{wrap_text(result.outcome_name, max_len=28, max_lines=5)}\n"
                f"{format_prob(result.path_probability)}"
            )

            # Add the final outcome node
            dot.node(
                outcome_id,
                label=label_text,
                shape="box",
                style="rounded,filled",
                fillcolor=outcome_fillcolor(result.outcome_category),
                color="black",
                image=icon_paths[outcome_icon_key(result.outcome_category)],
                labelloc="b",
                imagescale="true",
                width="2.9",
                height="1.2",
                fontsize="8.5",
                penwidth="1.2"
            )

            # Choose edge color based on outcome severity
            edge_color = (
                "darkgreen" if result.outcome_category == "SAFE"
                else "goldenrod" if result.outcome_category == "PARTIAL FAILURE"
                else "red"
            )

            # Connect the final branch endpoint to the outcome node
            dot.edge(
                parent_stub,
                outcome_id,
                arrowhead="none",
                color=edge_color,
                penwidth="1.2"
            )

        # Render the graph to a PNG image file
        png_path = dot.render(output_name, cleanup=True)

        # If image was created successfully, display it in the notebook
        if os.path.exists(png_path):
            display(Image(filename=png_path))
        else:
            print("Could not find generated ETA image. Please check Graphviz installation.")

        # Print the saved file location
        print(f"ETA diagram saved as: {png_path}")

        # Return the image file path
        return png_path

    # This method runs the full ETA summary workflow and returns the output tables
    def run_dashboard(self, image_path: Optional[str] = None) -> Dict[str, pd.DataFrame]:

        # Generate all event paths first
        path_results = self.generate_paths()

        # Sum probabilities of all SAFE paths
        safe_total = sum(p.path_probability for p in path_results if p.outcome_category == "SAFE")

        # Sum probabilities of all PARTIAL FAILURE paths
        partial_total = sum(p.path_probability for p in path_results if p.outcome_category == "PARTIAL FAILURE")

        # Sum probabilities of all CATASTROPHIC FAILURE paths
        catastrophic_total = sum(p.path_probability for p in path_results if p.outcome_category == "CATASTROPHIC FAILURE")

        # Print a heading line
        print("=" * 115)

        # Print the title in uppercase
        print(self.title.upper())

        # Print another heading line
        print("=" * 115)
        print()

        # Print the initiating event
        print(f"Initiating Event: {self.initiating_event}")

        # Print the total number of generated paths
        print(f"Total Number of Paths: {len(path_results)}")

        # Print overall safe probability
        print(f"Safe Outcome Probability: {format_prob(safe_total)}")

        # Print overall partial failure probability
        print(f"Partial Failure Probability: {format_prob(partial_total)}")

        # Print overall catastrophic failure probability
        print(f"Catastrophic Failure Probability: {format_prob(catastrophic_total)}")
        print()

        # Draw and display the event tree diagram
        self.visualize_event_tree(save_path=image_path)

        # Build the barrier summary table
        barrier_df = self.barrier_table()

        # Build the detailed path table
        path_df = self.path_table()

        # Build the grouped outcome summary table
        outcome_df = self.outcome_summary_table()

        # Build the category comparison table
        category_df = self.category_comparison_table()

        # Return all generated tables in one dictionary
        return {
            "barrier_probability_table": barrier_df,
            "path_outcome_table": path_df,
            "final_outcome_summary": outcome_df,
            "category_comparison": category_df,
        }

## Block 7: Define the Case-Specific Outcome Logic

This block defines how the final consequence is selected for each path.

The function receives a list of six True/False decisions representing:
- IDS success or failure
- authentication success or failure
- relay success or failure
- alarm success or failure
- operator success or failure
- shutdown success or failure

Based on these conditions, the function returns a final outcome description.

This is the logic that transforms a branch sequence into an engineering consequence.

In [23]:
# This function defines how the final outcome is decided for each path
def substation_outcome_logic(decisions: List[bool]) -> str:

    # Extract each barrier result from the decisions list
    ids_detects = decisions[0]        # Intrusion detection system result
    auth_works = decisions[1]         # Authentication result
    relay_works = decisions[2]        # Protection relay result
    alarm_works = decisions[3]        # Alarm system result
    operator_works = decisions[4]     # Operator action result
    shutdown_works = decisions[5]     # Shutdown system result

    # If intrusion is detected early, system remains safe immediately
    if ids_detects:
        return "Attack detected early - Safe operation maintained"

    # If IDS fails but authentication blocks the action, system stays safe
    if not ids_detects and auth_works:
        return "Unauthorized command rejected - Safe state maintained"

    # If both IDS and authentication fail, but relay and shutdown work,
    # the fault is still contained safely by protection systems
    if not ids_detects and not auth_works and relay_works and shutdown_works:
        return "Fault isolated by protection and shutdown system - Stable operation"

    # If shutdown fails but operator responds successfully, partial recovery happens
    if (
        not ids_detects
        and not auth_works
        and relay_works
        and alarm_works
        and operator_works
        and not shutdown_works
    ):
        return "Operator response successful but shutdown unavailable - Partial recovery"

    # If operator fails but automatic shutdown works, delayed safe recovery occurs
    if (
        not ids_detects
        and not auth_works
        and relay_works
        and alarm_works
        and not operator_works
        and shutdown_works
    ):
        return "Automatic shutdown successful despite operator failure - Delayed safe recovery"

    # If alarm fails but shutdown still works, system degrades but stays controlled
    if (
        not ids_detects
        and not auth_works
        and relay_works
        and not alarm_works
        and shutdown_works
    ):
        return "Alarm failure but automatic shutdown isolates event - Controlled degradation"

    # If relay fails but operator and shutdown succeed, system is manually recovered
    if (
        not ids_detects
        and not auth_works
        and not relay_works
        and operator_works
        and shutdown_works
    ):
        return "Manual containment after relay failure - Partial recovery"

    # If major protections fail including shutdown and operator, blackout occurs
    if (
        not ids_detects
        and not auth_works
        and not relay_works
        and not shutdown_works
        and not operator_works
    ):
        return "Major blackout and cascading failure"

    # If all systems fail including alarm, operator, and shutdown, worst-case occurs
    if (
        not ids_detects
        and not auth_works
        and not relay_works
        and not alarm_works
        and not operator_works
        and not shutdown_works
    ):
        return "Catastrophic substation failure with cascading grid impact"

    # Default case if no specific condition is matched
    return "Equipment damage and local outage"

## Block 8: Build the Smart Substation ETA Case

This block creates the actual case study instance.

Each barrier is assigned:
- a barrier name
- a success probability
- a barrier type

The six barriers together represent the defense-in-depth structure of the smart substation.

Finally, the `EventTree` object is created using:
- the initiating event
- the list of barriers
- the case study title
- the case-specific outcome logic

In [24]:
# This function builds and returns the complete ETA case study for the substation
def build_substation_eta_case() -> EventTree:

    # Define all barriers used in the event tree
    barriers = [

        # Cybersecurity barrier: detects abnormal or malicious activity
        ETABarrier("Intrusion detection system detects anomaly", 0.85, "SECURITY"),

        # Cybersecurity barrier: prevents unauthorized access
        ETABarrier("Access control / authentication works", 0.90, "SECURITY"),

        # Technical safety barrier: relay protects system from faults
        ETABarrier("Protection relay logic functions correctly", 0.92, "SAFETY"),

        # Technical safety barrier: alarm alerts system/operator
        ETABarrier("Alarm / supervisory response works", 0.88, "SAFETY"),

        # Human barrier: operator takes corrective action
        ETABarrier("Operator intervention succeeds", 0.80, "HUMAN"),

        # Final safety barrier: isolates or shuts down the system
        ETABarrier("Shutdown / isolation system works", 0.94, "SAFETY"),
    ]

    # Create and return the EventTree object using the defined inputs
    return EventTree(

        # Define the initiating event (starting point of the ETA)
        initiating_event="Malicious or faulty control command affects substation operation",

        # Attach the list of barriers to the tree
        barriers=barriers,

        # Provide a descriptive title for the analysis
        title="Cyber-Induced Protection Failure in Smart Power Grid Substation - Event Tree Analysis",

        # Attach the custom outcome logic function
        outcome_logic=substation_outcome_logic
    )

## Block 9: Run the ETA Model

This block executes the full Event Tree Analysis.

It:
- creates the ETA case
- runs the dashboard
- generates the event tree diagram
- stores all output tables in a dictionary

This is the main execution step of the notebook.

In [25]:
substation_eta = build_substation_eta_case()
eta_results = substation_eta.run_dashboard(image_path="smart_power_grid_substation_eta_horizontal")

CYBER-INDUCED PROTECTION FAILURE IN SMART POWER GRID SUBSTATION - EVENT TREE ANALYSIS

Initiating Event: Malicious or faulty control command affects substation operation
Total Number of Paths: 64
Safe Outcome Probability: P = 0.998
Partial Failure Probability: P = 0.002
Catastrophic Failure Probability: P = 0



ETA diagram saved as: smart_power_grid_substation_eta_horizontal.png


## Block 10: Display Final Output Tables

This block displays the four main ETA result tables:

1. Barrier probability table
2. Path outcome table
3. Final outcome summary table
4. Category comparison table

These tables help interpret the event tree in a structured numerical form.

In [17]:
eta_results["barrier_probability_table"]
eta_results["path_outcome_table"]
eta_results["final_outcome_summary"]
eta_results["category_comparison"]

,ID,Category,Total Probability
0,1,SAFE,P = 0.998
1,2,PARTIAL FAILURE,P = 0.002
2,3,CATASTROPHIC FAILURE,P = 0


## Final Interpretation of the Output

The event tree begins with a cyber-induced initiating event in a smart substation. It then tracks the success or failure of six barriers across cybersecurity, technical safety, and human response layers.

Because each barrier has two possible states, the model generates 64 possible paths. Each path probability is calculated by multiplying the probabilities along that branch sequence.

The final outcomes are classified into:
- SAFE
- PARTIAL FAILURE
- CATASTROPHIC FAILURE

The ETA diagram provides the visual accident progression, while the tables provide structured numerical results.

This makes the model useful for showing:
- how layered defenses reduce risk
- how failures propagate through the system
- which combinations lead to safe recovery
- which combinations lead to severe consequences